In [1]:
import bar_chart_race_ as bcr
import pandas as pd
bcr = bcr.bar_chart_race_plotly(orientation='h', period_length=1000, 
                                interpolate_period=True, 
                                steps_per_period=2, n_bars=20, 
                                autorange_vals = True,
                                val_ax_label = '-log10(p.adj)',
                                title='Locally enriched pathways per sliding window position')
bcr.make_animation()

creating frames: 100%|██████████| 1000/1000 [00:02<00:00, 424.23it/s]


In [ ]:
import numpy as np
import pandas as pd
df = pd.DataFrame({'A': [1, np.nan, 3, 4], 'B': [3, 4, np.nan, np.nan], 'C': [np.nan, 9, 3, 1]})
df = df.fillna(0)
df = df.astype('int32')
print(df)
# steps_per_period = 2
# df.index = df.index * steps_per_period
# new_index = range(df.index[-1]+1)
# df= df.reindex(new_index)
# df = df.interpolate()
# print(df)
# df = df.rank(axis=1, method='first')
# df[df>2] = np.nan
# print(df)
# #df_out = pd.DataFrame({1: ['A', 'B', 'A', 'C'], 2:  ['B', 'C', 'C', 'A']})
# #print(df_out)

# s = df.stack().reset_index()
# df2 = pd.DataFrame(s)
# df2 = df2.pivot(index='level_0', columns=0, values='level_1')
# print(df2)

[0 4 9]
['a' 'b']
   A  B  C
0  1  3  0
1  0  4  9
2  3  0  3
3  4  0  1


In [ ]:
import bar_chart_race_ as bcr
import pandas as pd
import plotly.graph_objects as go
import plotly
import numpy as np
import os

def get_pw_df_and_lut(pw_data_fpath):
    if os.path.exists('./data/pathway_data_idxed.csv') and os.path.exists('./data/pathway_names.txt'):
        df = pd.read_csv('./data/pathway_data_idxed.csv', index_col='global_index')
        with open('./data/pathway_names.txt') as f:
            pw_names = f.readlines()
        return df, pw_names

    df = pd.read_csv(pw_data_fpath, index_col='global_index')
    df = df.drop('p.adj', axis=1)
    pw_names = df["pathway.name"].unique()
    pw_idx_map = {name: i for i, name in enumerate(pw_names)}
    df["pathway.idx"] = df["pathway.name"].map(pw_idx_map)
    df = df.drop(columns=["pathway.name"])
    df.to_csv(os.path.join(os.path.dirname(pw_data_fpath),'pathway_data_idxed.csv'))
    with open(os.path.join(os.path.dirname(pw_data_fpath),'pathway_names.txt'), 'w') as f:
        for pw in pw_names: f.write(pw+'\n')
    return df, pw_names

In [ ]:
def get_wide_df_and_lut():
    wide_df_fpath = './data/pathway_data_wide.csv'
    if os.path.exists(wide_df_fpath) and os.path.exists('./data/pathway_names.txt'):
        df_wide = pd.read_csv(wide_df_fpath, index_col='window')
        with open('./data/pathway_names.txt') as f:
            pw_names = f.readlines()
    else: 
        df, pw_names = get_pw_df_and_lut('./data/pathway_data.csv')
        df_wide = pd.pivot_table(df, values='-log10(p.adj)', index='window', columns='pathway.idx')
        df_wide = df_wide.fillna(0)
        df_wide.to_csv(wide_df_fpath)
    return df_wide, pw_names

def get_plot_data():
    df_wide, pw_names = get_wide_df_and_lut()
    n_bars = 20
    steps_per_period = 2

    df_wide_idx = df_wide.index
    if df_wide.index[0] == 1:
        df_wide_idx -= 1

    df_wide.index = df_wide_idx * steps_per_period
    new_index = range(df_wide.index[-1]+1)
    df_wide = df_wide.reindex(new_index)

    df_vals = df_wide.interpolate()

    df_ranks_wide = df_vals.rank(axis=1, method='first', ascending=False)
    df_ranks_wide[df_ranks_wide > n_bars] = np.nan

    ser = df_ranks_wide.stack().reset_index()

    df_ser = pd.DataFrame(ser).astype('int32')

    df_ranks = df_ser.pivot(index='window', columns=0, values='level_1')
    return df_vals, df_ranks, pw_names

df_vals, df_ranks, pw_names = get_plot_data()

#df_vals.iloc[0, :n_bars].values
#df_ranks.iloc[0].values

In [ ]:
bar_textposition = 'outside'
orientation='h'
bar_size=.95 
bar_texttemplate=None
period_length=500
end_period_pause=0
period_label=True
period_template=None
period_summary_func=None
perpendicular_bar_func=None
colors=None
title='local pathway enrichment'
bar_size=.95 
bar_texttemplate=None
bar_label_font=None
tick_label_font=None 
hovertemplate=None 
slider=True
scale='linear' 
bar_kwargs={}
layout_kwargs=None
write_html_kwargs=None 
filter_column_colors=False




bar = go.Bar(
    x=bar_vals, 
    y=bar_labels, 
    width=bar_size, 
    textposition=bar_textposition,
    texttemplate=bar_texttemplate, 
    orientation=orientation, 
    marker_color='viridis', 
    insidetextfont=bar_label_font, 
    cliponaxis=False, 
    outsidetextfont=bar_label_font, 
    hovertemplate=hovertemplate, 
    **(bar_kwargs or {})  # Use empty dict if bar_kwargs is None
)

go.Figure([bar])


In [ ]:
class RBC:
    def __init__(self, df_vals, df_ranks):
        self.df_vals = df_vals
        self.df_ranks = df_ranks

    def get_frames(self):
        frames = []
        slider_steps = []
        
        cols = self.df_vals.columns.values.copy()

        for i in range(len(self.df_vals)):
            bar_locs = self.df_ranks.iloc[i].values
            bar_vals = self.df_vals.iloc[i, :n_bars].values

            colors = self.bar_colors
            #bar_locs = bar_locs + np.random.rand(len(bar_locs)) / 10000 # done to prevent stacking of bars

            label_axis = dict(tickmode='array', tickvals=bar_locs, ticktext=cols, 
                                tickfont=self.tick_label_font)

            label_axis['range'] = self.ylimit if self.orientation == 'h' else self.xlimit
            if self.orientation == 'v':
                label_axis['tickangle'] = -90

            value_axis = dict(showgrid=True, type=self.scale)#, tickformat=',.0f')
            value_axis['range'] = self.xlimit if self.orientation == 'h' else self.ylimit

            bar = go.Bar(x=bar_vals, y=bar_locs, width=self.bar_size, textposition=self.bar_textposition,
                            texttemplate=self.bar_texttemplate, orientation=self.orientation, 
                            marker_color=colors, insidetextfont=self.bar_label_font, 
                            cliponaxis=False, outsidetextfont=self.bar_label_font, 
                            hovertemplate=self.hovertemplate, **self.bar_kwargs)

            data = [bar]
            xaxis, yaxis = (value_axis, label_axis) if self.orientation == 'h' \
                                else (label_axis, value_axis)
            
            annotations = self.get_annotations(i) # 
            if self.slider and i % self.steps_per_period == 0:
                slider_steps.append(
                            {"args": [[i],
                                {"frame": {"duration": self.duration, "redraw": False},
                                    "mode": "immediate",
                                    "fromcurrent": True,
                                    "transition": {"duration": self.duration}
                                }],
                            "label": self.get_period_label_text(i), 
                            "method": "animate"})
            layout = go.Layout(xaxis=xaxis, yaxis=yaxis, annotations=annotations, 
                                margin={'l': 150}, **self.layout_kwargs)
            if self.perpendicular_bar_func:
                pbar = self.get_perpendicular_bar(bar_vals, i, layout)
                layout.update(shapes=[pbar], overwrite=True)
            frames.append(go.Frame(data=data, layout=layout, name=i))

        return frames, slider_steps
    


In [ ]:
# no wide data format interpolation
df, pw_names = get_pw_df('./data/pathway_data.csv')
#df = pd.DataFrame({'window': [1]*5+[2]*5, 'obj': np.random.randint(1,5,5)+np.random.randint(1,5,5), 'v': -np.sort(-np.random.rand(10)*10)})
print(df)
steps_per_period = 2
# df.index = df.index * steps_per_period
# print(df.index)
# new_index = range(df.index[-1] + 1)
# df= df.reindex(new_index)
df[['window']] = df[['window']] * steps_per_period

print(df)

from typing import Optional, Iterable
import csv
from tqdm import tqdm

def generate_interpolated_race_frames(
    df: pd.DataFrame,
    steps_per_period: int = 10,
    value_col: str = "-log10(p.adj)",
    window_col: str = "window",
    pathway_col: str = "pathway.idx",
    time_col: str = "time",                     # name for fractional window/time
    frame_col: str = "frame",                   # sequential frame index
    fill_method: str = "none",                  # "none" | "ffill" | "bfill" | "carry" - see below
    out_path: Optional[str] = "race_frames.csv",# path to write streaming output; None => return DataFrame (may be large)
    index_dtype: Optional[type] = int
) -> Optional[pd.DataFrame]:
    """
    Stream/generate interpolated frames for a race bar.

    - df: input dataframe with at least columns [window_col, pathway_col, value_col]
    - steps_per_period: integer number of steps between consecutive integer windows
    - fill_method:
        * "none": interpolate only if both windows have values; otherwise pathway absent (NaN).
        * "ffill": if value missing at start, carry previous known value forward (requires prior data).
        * "carry": if present in one of the two windows, carry that value across the interpolated steps.
    - top_n: keep only top_n per frame (recommended to reduce size for race chart).
    - out_path: CSV path to write results. If None, returns a DataFrame (may be large).
    Returns DataFrame if out_path is None, else writes CSV and returns None.
    """

    # --- Basic checks / normalize types ---
    df = df.copy()
    if index_dtype is not None:
        df[pathway_col] = df[pathway_col].astype(index_dtype)
    df[window_col] = df[window_col].astype(float)  # windows might be ints but use float for generality
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')

    # Group into mapping: window -> {pathway: value}
    windows = sorted(df[window_col].unique())
    # build dictionary per window
    values_by_window = {}
    for w, group in df.groupby(window_col):
        values_by_window[float(w)] = dict(zip(group[pathway_col].tolist(), group[value_col].tolist()))

    # precompute set of all pathways
    all_pathways = sorted(df[pathway_col].unique())

    # helper to get value at given window for a pathway (or np.nan)
    def val_at(window: float, pathway):
        return values_by_window.get(window, {}).get(pathway, np.nan)

    # If user selected 'ffill' or 'bfill' we need a last-known map for ffill and lookahead for bfill.
    last_known = {pw: np.nan for pw in all_pathways}
    if fill_method == "bfill":
        # precompute next-known values per window for each pathway
        # We'll create a map: next_val_after[w][pw] = next known value at window >= w
        # Simpler: compute next known for each (pathway -> list of (window,value)), then during pair use it.
        next_known = {}
        for pw in all_pathways:
            rows = df[df[pathway_col] == pw].sort_values(window_col)
            if rows.empty:
                next_known[pw] = []
            else:
                next_known[pw] = list(zip(rows[window_col].astype(float).tolist(), rows[value_col].tolist()))

    # prepare output (stream)
    header = [frame_col, time_col, pathway_col, value_col, "rank"]
    if out_path:
        f = open(out_path, "w", newline="")
        writer = csv.writer(f)
        writer.writerow(header)
    else:
        rows_out = []

    frame_idx = 0
    # iterate consecutive window pairs
    for i in tqdm(range(len(windows)-1), desc="window pairs"):
        w0 = float(windows[i])
        w1 = float(windows[i+1])
        map0 = values_by_window.get(w0, {})
        map1 = values_by_window.get(w1, {})

        # union of pathways present in either window (faster than iterating all 600 every time)
        pws_union = set(map0.keys()) | set(map1.keys())

        # For each step in [0 .. steps_per_period-1], produce a fractional frame corresponding to t = w0 + step/steps_per_period
        for step in range(steps_per_period):
            frac = step / float(steps_per_period)
            t = w0 + frac  # fractional time index -- you can use (w0 + frac*(w1-w0)) if windows are non-unit spaced
            # collect entries for this frame
            pws = []
            vals = []

            # compute values
            for pw in pws_union:
                v0 = map0.get(pw, np.nan)
                v1 = map1.get(pw, np.nan)

                if not np.isnan(v0) and not np.isnan(v1):
                    v = v0 + (v1 - v0) * frac
                else:
                    # some missing cases
                    if fill_method == "none":
                        v = np.nan
                    elif fill_method == "carry":
                        # if only one side present, carry it
                        if not np.isnan(v0):
                            v = v0
                        elif not np.isnan(v1):
                            v = v1
                        else:
                            v = np.nan
                    elif fill_method == "ffill":
                        # use last_known (from previous windows)
                        prev = last_known.get(pw, np.nan)
                        v = prev if not np.isnan(prev) else np.nan
                    elif fill_method == "bfill":
                        # Attempt to find next known value in next_known list with window >= w1
                        # fallback to NaN
                        next_list = next_known.get(pw, [])
                        # find first entry with window >= w1
                        v = np.nan
                        for wn, vv in next_list:
                            if wn >= w1:
                                v = vv
                                break
                    else:
                        v = np.nan

                if not np.isnan(v):
                    pws.append(pw)
                    vals.append(v)

            if len(vals) == 0:
                frame_idx += 1
                continue

            # compute ranks: higher value => rank 1
            vals_arr = np.array(vals)
            order_desc = np.argsort(vals_arr)[::-1]
            ranks = np.empty(len(vals_arr), dtype=int)
            ranks[order_desc] = np.arange(1, len(vals_arr)+1)

            # prepare output rows; optionally keep top_n
            if top_n is not None:
                # select indices of top_n by value
                k = min(top_n, len(vals_arr))
                top_idx = order_desc[:k]
            else:
                top_idx = np.arange(len(vals_arr))

            for idx in top_idx:
                out_row = [frame_idx, t, pws[idx], vals_arr[idx], int(ranks[idx])]
                if out_path:
                    writer.writerow(out_row)
                else:
                    rows_out.append(out_row)

            frame_idx += 1

        # update last_known map for ffill
        if fill_method == "ffill":
            for pw, v in map1.items():
                if not np.isnan(v):
                    last_known[pw] = v

    # Optionally also include the final window as a full final frame (step == steps_per_period)
    # If you want that, add a block to emit window = last windows[-1]
    if out_path:
        f.close()
        print(f"Frames streamed to {out_path}")
        return None
    else:
        df_out = pd.DataFrame(rows_out, columns=header)
        return df_out



              rank  -log10(p.adj)  window  pathway.idx
global_index                                          
1                1      24.073566       1            0
2                2      14.812193       1            1
3                3       6.631502       1            2
4                4       5.632760       1            3
5                5       5.167984       1            4
...            ...            ...     ...          ...
136496          16       3.856456    6825          174
136497          17       3.746541    6825           60
136498          18       3.299264    6825           73
136499          19       3.259919    6825          177
136500          20       3.259919    6825          176

[136500 rows x 4 columns]
              rank  -log10(p.adj)  window  pathway.idx
global_index                                          
1                1      24.073566       2            0
2                2      14.812193       2            1
3                3       6.631502     

In [ ]:
def prepare_data(df, sort='desc', n_bars=20, interpolate_period=True, 
                      steps_per_period=10):

    df[['window']] = df[['window']] * steps_per_period
    new_index = range(df_values.index[-1] + 1)
    print(df[['window']])
    df.index = df.index * steps_per_period
    new_index = range(df_values.index[-1] + 1)
    df_values = df_values.reindex(new_index)
    if interpolate_period:
        if df_values.iloc[:, 0].dtype.kind == 'M':
            first, last = df_values.iloc[[0, -1], 0]
            dr = pd.date_range(first, last, periods=len(df_values))
            df_values.iloc[:, 0] = dr
        else:
            df_values.iloc[:, 0] = df_values.iloc[:, 0].interpolate()
    # else:
    #     df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
    
    df_values = df_values.set_index(df_values.columns[0])
    if compute_ranks:
        df_ranks = df_values.rank(axis=1, method='first', ascending=False).clip(upper=n_bars + 1)
        if sort == 'desc':
            df_ranks = n_bars + 1 - df_ranks
        df_ranks = df_ranks.interpolate()
    
    df_values = df_values.interpolate()
    return df_interp


In [ ]:
# look at specific frame
import bar_chart_race_ as bcr
import pandas as pd
import plotly.graph_objects as go
import plotly
import numpy as np

df = pd.read_csv('./data/covid19.csv', index_col='date', parse_dates=True)
#bcr.bar_chart_race_plotly(df, filename='covid19_plotly.html')
bcr = bcr.bar_chart_race_plotly(df, orientation='h', period_length=200)
#print(bcr.df_values.head())
#print(bcr.df_ranks.head())
print('bcr.n_bars= ', bcr.n_bars)
print('xlimit, ylimit: ', bcr.xlimit, bcr.ylimit)

frames = []
slider_steps = []

i=300
bar_locs = bcr.df_ranks.iloc[i].values
top_filt = (bar_locs >= 0) & (bar_locs < bcr.n_bars + 1)
bar_vals = bcr.df_values.iloc[i].values
print('bar_locs')
print(bar_locs)

bar_vals[bar_locs == 0] = 0
bar_vals[bar_locs == bcr.n_bars + 1] = 0
print('bar_vals')
print(bar_vals)

# bcr.set_value_limit(bar_vals) # plotly bug? not updating range

cols = bcr.df_values.columns.values.copy()

cols[bar_locs == 0] = ' '
print('cols')
print(cols)
colors = bcr.bar_colors
bar_locs = bar_locs + np.random.rand(len(bar_locs)) / 10_000 # done to prevent stacking of bars
x, y = (bar_vals, bar_locs) if bcr.orientation == 'h' else (bar_locs, bar_vals)

print('x: ', x)
print('y: ', y)
label_axis = dict(tickmode='array', tickvals=bar_locs, ticktext=cols, 
                    tickfont=bcr.tick_label_font)

label_axis['range'] = bcr.ylimit if bcr.orientation == 'h' else bcr.xlimit
if bcr.orientation == 'v':
    label_axis['tickangle'] = -90

value_axis = dict(showgrid=True, type=bcr.scale)#, tickformat=',.0f')
value_axis['range'] = bcr.xlimit if bcr.orientation == 'h' else bcr.ylimit

bar = go.Bar(x=x, y=y, width=bcr.bar_size, textposition=bcr.bar_textposition,
                texttemplate=bcr.bar_texttemplate, orientation=bcr.orientation, 
                marker_color=colors, insidetextfont=bcr.bar_label_font, 
                cliponaxis=False, outsidetextfont=bcr.bar_label_font, 
                hovertemplate=bcr.hovertemplate, **bcr.bar_kwargs)

data = [bar]
xaxis, yaxis = (value_axis, label_axis) if bcr.orientation == 'h' else (label_axis, value_axis)

print('xaxis: ', xaxis)
print('yaxis: ', yaxis)
annotations = bcr.get_annotations(i)
if bcr.slider and i % bcr.steps_per_period == 0:
    slider_steps.append(
                {"args": [[i],
                    {"frame": {"duration": bcr.duration, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True,
                        "transition": {"duration": bcr.duration}
                    }],
                "label": bcr.get_period_label_text(i), 
                "method": "animate"})
layout = go.Layout(xaxis=xaxis, yaxis=yaxis, annotations=annotations, 
                    margin={'l': 150}, **bcr.layout_kwargs)
frames.append(go.Frame(data=data, layout=layout, name=i))

fig = go.Figure(data=data, layout=layout, frames=frames)
fig.show()
#bcr.make_animation()


bcr.n_bars=  20
xlimit, ylimit:  None (0.2, 20.8)
bar_locs
[11.  8.  4. 18. 16. 12.  2.  6. 17.  3. 20.  1. 13.  5. 19.  9. 10.  7.
 15. 14.]
bar_vals
[2.890e+02 9.200e+01 5.400e+01 3.296e+03 1.997e+03 3.420e+02 2.000e+01
 8.700e+01 2.378e+03 2.200e+01 9.134e+03 8.000e+00 5.470e+02 7.600e+01
 5.138e+03 1.050e+02 2.310e+02 9.200e+01 1.582e+03 7.610e+02]
cols
['Belgium' 'Brazil' 'Canada' 'China' 'France' 'Germany' 'India'
 'Indonesia' 'Iran' 'Ireland' 'Italy' 'Mexico' 'Netherlands' 'Portugal'
 'Spain' 'Sweden' 'Switzerland' 'Turkey' 'USA' 'United Kingdom']
x:  [2.890e+02 9.200e+01 5.400e+01 3.296e+03 1.997e+03 3.420e+02 2.000e+01
 8.700e+01 2.378e+03 2.200e+01 9.134e+03 8.000e+00 5.470e+02 7.600e+01
 5.138e+03 1.050e+02 2.310e+02 9.200e+01 1.582e+03 7.610e+02]
y:  [11.0000346   8.00009468  4.0000145  18.0000364  16.00006247 12.00009464
  2.00003569  6.00001231 17.00006148  3.00004895 20.00006213  1.00005417
 13.00001951  5.00003292 19.00005597  9.0000137  10.00005386  7.00000571
 15.0000

/home/chr/Uni/Master/5.Semester/FGM/Python/Bar_Chart_Race_Package_Test/bar_chart_race_modified/bar_chart_race/_utils.py:111: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [ ]:
# appending frames, not working try
import bar_chart_race_modified.bar_chart_race as bcr
import pandas as pd
import plotly.graph_objects as go
import plotly
import numpy as np


#df = bcr.load_dataset('covid19')
df = pd.read_csv('./bar_chart_race_modified/data/covid19.csv', index_col='date', parse_dates=True)
bcr = bcr.bar_chart_race_plotly(df, orientation='h', period_length=200)

frames = []
slider_steps = []

steps = int(np.ceil(len(bcr.df_values[:10].index)/10))
for i in range(steps):
    for j in range(np.max([10,len(bcr.df_values[i:])])):

        bar_locs = bcr.df_ranks.iloc[i].values
        #top_filt = (bar_locs >= 0) & (bar_locs < bcr.n_bars + 1)
        bar_vals = bcr.df_values.iloc[i].values

        #bar_vals[bar_locs == 0] = 0
        #bar_vals[bar_locs == bcr.n_bars + 1] = 0

        # bcr.set_value_limit(bar_vals) # plotly bug? not updating range

        cols = bcr.df_values.columns.values.copy()

        #cols[bar_locs == 0] = ' '

        colors = bcr.bar_colors
        bar_locs = bar_locs + np.random.rand(len(bar_locs)) / 10_000 # done to prevent stacking of bars
        x, y = (bar_vals, bar_locs) if bcr.orientation == 'h' else (bar_locs, bar_vals)


        label_axis = dict(tickmode='array', tickvals=bar_locs, ticktext=cols, 
                            tickfont=bcr.tick_label_font)

        label_axis['range'] = bcr.ylimit if bcr.orientation == 'h' else bcr.xlimit
        if bcr.orientation == 'v':
            label_axis['tickangle'] = -90

        value_axis = dict(showgrid=True, type=bcr.scale)#, tickformat=',.0f')
        value_axis['range'] = bcr.xlimit if bcr.orientation == 'h' else bcr.ylimit

        bar = go.Bar(x=x, y=y, width=bcr.bar_size, textposition=bcr.bar_textposition,
                        texttemplate=bcr.bar_texttemplate, orientation=bcr.orientation, 
                        marker_color=colors, insidetextfont=bcr.bar_label_font, 
                        cliponaxis=False, outsidetextfont=bcr.bar_label_font, 
                        hovertemplate=bcr.hovertemplate, **bcr.bar_kwargs)

        data = [bar]
        xaxis, yaxis = (value_axis, label_axis) if bcr.orientation == 'h' else (label_axis, value_axis)

        annotations = bcr.get_annotations(i)
        if bcr.slider and i % bcr.steps_per_period == 0:
            slider_steps.append(
                    {"args": [[i],
                        {"frame": {"duration": bcr.duration, "redraw": False},
                            "mode": "immediate",
                            "fromcurrent": True,
                            "transition": {"duration": bcr.duration}
                        }],
                    "label": bcr.get_period_label_text(i), 
                    "method": "animate"})
        layout = go.Layout(xaxis=xaxis, yaxis=yaxis, annotations=annotations, 
                        margin={'l': 150}, **bcr.layout_kwargs)
        frames.append(go.Frame(data=data, layout=layout, name=i))

    #fig.show()
    if i == 0:
        frames, slider_steps = frames, slider_steps
        data = frames[0].data
        layout = frames[0].layout
        layout.title = bcr.title
        layout.updatemenus = [dict(
            type="buttons",
            direction = "left",
            x=1, 
            y=1.02,
            xanchor='right',
            yanchor='bottom',
            buttons=[dict(label="Play",
                          method="animate",
                          # redraw must be true for bar plots
                          args=[None, {"frame": {"duration": bcr.duration, "redraw": True},
                                        "fromcurrent": True
                                    }]),
                     dict(label="Pause",
                          method="animate",
                          args=[[None], {"frame": {"duration": 0, "redraw": False},
                                         "mode": "immediate",
                                         "transition": {"duration": 0}}]),
                     ]
                     )]

        sliders_dict = {
                        "active": 0,
                        "yanchor": "top",
                        "xanchor": "left",
                        "currentvalue": {
                            # "font": {"size": 20},
                            # "prefix": '', # allow user to set
                            "visible": False, # just repeats period label
                            # "xanchor": "right"
                        },
                        "transition": {"duration": bcr.duration, "easing": "cubic-in-out"},
                        "pad": {"b": 10, "t": 50},
                        "len": 0.88,
                        "x": 0.05,
                        "y": 0,
                        "steps": slider_steps
                    }
        if bcr.slider:
            layout.sliders = [sliders_dict]

        fig = go.Figure(data=data, layout=layout, frames=frames[1:])
        fig.show()
        


In [ ]:
import bar_chart_race_modified.bar_chart_race as bcr_pckg
import pandas as pd
import plotly.graph_objects as go
import plotly
import numpy as np

def create_bcr():
    #df = bcr.load_dataset('covid19')
    df = pd.read_csv('./bar_chart_race_modified/data/covid19.csv', index_col='date', parse_dates=True)

    bcr = bcr_pckg.bar_chart_race_plotly(df, orientation='h', period_length=200)
    return bcr

def initialize_bcr(bcr):

    print('bcr.n_bars= ', bcr.n_bars)
    print('xlimit, ylimit: ', bcr.xlimit, bcr.ylimit)

    frames = []
    slider_steps = []

    for i in range(10):
        bar_locs = bcr.df_ranks.iloc[i].values
        #top_filt = (bar_locs >= 0) & (bar_locs < bcr.n_bars + 1)
        bar_vals = bcr.df_values.iloc[i].values
        #print('bar_locs')
        #print(bar_locs)

        #bar_vals[bar_locs == 0] = 0
        #bar_vals[bar_locs == bcr.n_bars + 1] = 0
        #print('bar_vals')
        #print(bar_vals)

        # bcr.set_value_limit(bar_vals) # plotly bug? not updating range

        cols = bcr.df_values.columns.values.copy()

        #cols[bar_locs == 0] = ' '
        #print('cols')
        #print(cols)
        colors = bcr.bar_colors
        bar_locs = bar_locs + np.random.rand(len(bar_locs)) / 10_000 # done to prevent stacking of bars
        x, y = (bar_vals, bar_locs) if bcr.orientation == 'h' else (bar_locs, bar_vals)

        #print('x: ', x)
        #print('y: ', y)
        label_axis = dict(tickmode='array', tickvals=bar_locs, ticktext=cols, 
                            tickfont=bcr.tick_label_font)

        label_axis['range'] = bcr.ylimit if bcr.orientation == 'h' else bcr.xlimit
        if bcr.orientation == 'v':
            label_axis['tickangle'] = -90

        value_axis = dict(showgrid=True, type=bcr.scale)#, tickformat=',.0f')
        value_axis['range'] = bcr.xlimit if bcr.orientation == 'h' else bcr.ylimit

        bar = go.Bar(x=x, y=y, width=bcr.bar_size, textposition=bcr.bar_textposition,
                        texttemplate=bcr.bar_texttemplate, orientation=bcr.orientation, 
                        marker_color=colors, insidetextfont=bcr.bar_label_font, 
                        cliponaxis=False, outsidetextfont=bcr.bar_label_font, 
                        hovertemplate=bcr.hovertemplate, **bcr.bar_kwargs)

        data = [bar]
        xaxis, yaxis = (value_axis, label_axis) if bcr.orientation == 'h' else (label_axis, value_axis)

        #print('xaxis: ', xaxis)
        #print('yaxis: ', yaxis)
        annotations = bcr.get_annotations(i)
        if bcr.slider and i % bcr.steps_per_period == 0:
            slider_steps.append(
                    {"args": [[i],
                        {"frame": {"duration": bcr.duration, "redraw": False},
                            "mode": "immediate",
                            "fromcurrent": True,
                            "transition": {"duration": bcr.duration}
                        }],
                    "label": bcr.get_period_label_text(i), 
                    "method": "animate"})
        layout = go.Layout(xaxis=xaxis, yaxis=yaxis, annotations=annotations, 
                        margin={'l': 150}, **bcr.layout_kwargs)
        frames.append(go.Frame(data=data, layout=layout, name=i))


    frames, slider_steps = frames, slider_steps
    data = frames[0].data
    layout = frames[0].layout
    layout.title = bcr.title
    layout.updatemenus = [dict(
        type="buttons",
        direction = "left",
        x=1, 
        y=1.02,
        xanchor='right',
        yanchor='bottom',
        buttons=[dict(label="Play",
                        method="animate",
                        # redraw must be true for bar plots
                        args=[None, {"frame": {"duration": bcr.duration, "redraw": True},
                                    "fromcurrent": True
                                }]),
                    dict(label="Pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False},
                                        "mode": "immediate",
                                        "transition": {"duration": 0}}]),
                    ]
                    )]

    sliders_dict = {
                    "active": 0,
                    "yanchor": "top",
                    "xanchor": "left",
                    "currentvalue": {
                        # "font": {"size": 20},
                        # "prefix": '', # allow user to set
                        "visible": False, # just repeats period label
                        # "xanchor": "right"
                    },
                    "transition": {"duration": bcr.duration, "easing": "cubic-in-out"},
                    "pad": {"b": 10, "t": 50},
                    "len": 0.88,
                    "x": 0.05,
                    "y": 0,
                    "steps": slider_steps
                }
    if bcr.slider:
        layout.sliders = [sliders_dict]

    fig = go.Figure(data=data, layout=layout, frames=frames[1:])
    return fig

bcr = create_bcr()
fig = initialize_bcr(bcr)
fig.show()
print(fig.frames)
print(len(fig.frames))
steps = int(np.ceil(len(bcr.df_values.index)/10))
for i in range(1, steps):
    for j in range(np.max([10,len(bcr.df_values[i:])])):

        bar_locs = bcr.df_ranks.iloc[i].values
        #top_filt = (bar_locs >= 0) & (bar_locs < bcr.n_bars + 1)
        bar_vals = bcr.df_values.iloc[i].values
        #print('bar_locs')
        #print(bar_locs)

        #bar_vals[bar_locs == 0] = 0
        #bar_vals[bar_locs == bcr.n_bars + 1] = 0
        #print('bar_vals')
        #print(bar_vals)

        # bcr.set_value_limit(bar_vals) # plotly bug? not updating range

        cols = bcr.df_values.columns.values.copy()

        #cols[bar_locs == 0] = ' '
        #print('cols')
        #print(cols)
        colors = bcr.bar_colors
        bar_locs = bar_locs + np.random.rand(len(bar_locs)) / 10_000 # done to prevent stacking of bars
        x, y = (bar_vals, bar_locs) if bcr.orientation == 'h' else (bar_locs, bar_vals)

        #print('x: ', x)
        #print('y: ', y)
        label_axis = dict(tickmode='array', tickvals=bar_locs, ticktext=cols, 
                            tickfont=bcr.tick_label_font)

        label_axis['range'] = bcr.ylimit if bcr.orientation == 'h' else bcr.xlimit
        if bcr.orientation == 'v':
            label_axis['tickangle'] = -90

        value_axis = dict(showgrid=True, type=bcr.scale)#, tickformat=',.0f')
        value_axis['range'] = bcr.xlimit if bcr.orientation == 'h' else bcr.ylimit

        bar = go.Bar(x=x, y=y, width=bcr.bar_size, textposition=bcr.bar_textposition,
                        texttemplate=bcr.bar_texttemplate, orientation=bcr.orientation, 
                        marker_color=colors, insidetextfont=bcr.bar_label_font, 
                        cliponaxis=False, outsidetextfont=bcr.bar_label_font, 
                        hovertemplate=bcr.hovertemplate, **bcr.bar_kwargs)

        data = [bar]
        xaxis, yaxis = (value_axis, label_axis) if bcr.orientation == 'h' else (label_axis, value_axis)

        #print('xaxis: ', xaxis)
        #print('yaxis: ', yaxis)
        annotations = bcr.get_annotations(i)
        layout = go.Layout(xaxis=xaxis, yaxis=yaxis, annotations=annotations, 
                        margin={'l': 150}, **bcr.layout_kwargs)
        if bcr.slider and i % bcr.steps_per_period == 0:
            slider_steps.append(
                    {"args": [[i],
                        {"frame": {"duration": bcr.duration, "redraw": False},
                            "mode": "immediate",
                            "fromcurrent": True,
                            "transition": {"duration": bcr.duration}
                        }],
                    "label": bcr.get_period_label_text(i), 
                    "method": "animate"})
            sliders_dict = {
                        "active": 0,
                        "yanchor": "top",
                        "xanchor": "left",
                        "currentvalue": {
                            # "font": {"size": 20},
                            # "prefix": '', # allow user to set
                            "visible": False, # just repeats period label
                            # "xanchor": "right"
                        },
                        "transition": {"duration": bcr.duration, "easing": "cubic-in-out"},
                        "pad": {"b": 10, "t": 50},
                        "len": 0.88,
                        "x": 0.05,
                        "y": 0,
                        "steps": slider_steps
                    }

        layout.sliders = [sliders_dict]
        frames = list(fig.frames)
        frames.append(go.Frame(data=data, layout=layout, name=i))
        fig.frames = frames
        #fig.show()        


bcr.n_bars=  20
xlimit, ylimit:  None (0.2, 20.8)


/home/chr/Uni/Master/5.Semester/FGM/Python/Bar_Chart_Race_Package_Test/bar_chart_race_modified/bar_chart_race/_utils.py:111: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



(Frame({
    'data': [{'cliponaxis': False,
              'hovertemplate': '%{y} - %{x:,.0f}<extra></extra>',
              'insidetextfont': {'size': 12},
              'marker': {'color': array(['#2E91E5', '#E15F99', '#1CA71C', '#FB0D0D', '#DA16FF', '#222A2A',
                                         '#B68100', '#750D86', '#EB663B', '#511CFB', '#00A08B', '#FB00D1',
                                         '#FC0080', '#B2828D', '#6C7C32', '#778AAE', '#862A16', '#A777F1',
                                         '#620042', '#1616A7'], dtype=object)},
              'opacity': 0.8,
              'orientation': 'h',
              'outsidetextfont': {'size': 12},
              'textposition': 'outside',
              'texttemplate': '%{x:,.0f}',
              'type': 'bar',
              'width': 0.95,
              'x': array([       nan,        nan,        nan, 2.7199e+03, 2.0000e+00,        nan,
                                 nan,        nan, 1.9700e+01,        nan, 1.2500e+01,       

ValueError: 
    Invalid element(s) received for the 'data' property of frame
        Invalid elements include: [Frame({
    'data': [{'cliponaxis': False,
              'hovertemplate': '%{y} - %{x:,.0f}<extra></extra>',
              'insidetextfont': {'size': 12},
              'marker': {'color': array(['#2E91E5', '#E15F99', '#1CA71C', '#FB0D0D', '#DA16FF', '#222A2A',
                                         '#B68100', '#750D86', '#EB663B', '#511CFB', '#00A08B', '#FB00D1',
                                         '#FC0080', '#B2828D', '#6C7C32', '#778AAE', '#862A16', '#A777F1',
                                         '#620042', '#1616A7'], dtype=object)},
              'opacity': 0.8,
              'orientation': 'h',
              'outsidetextfont': {'size': 12},
              'textposition': 'outside',
              'texttemplate': '%{x:,.0f}',
              'type': 'bar',
              'width': 0.95,
              'x': array([       nan,        nan,        nan, 2.7199e+03, 2.00

In [ ]:
import bar_chart_race_modified.bar_chart_race as bcr
import pandas as pd
df = pd.read_csv('./bar_chart_race_modified/data/covid19.csv', index_col='date', parse_dates=True)
bcr = bcr.bar_chart_race_plotly(df, orientation='h', period_length=500)
bcr.make_animation()

/home/chr/Uni/Master/5.Semester/FGM/Python/Bar_Chart_Race_Package_Test/bar_chart_race_modified/bar_chart_race/_utils.py:111: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

